# Task 1.2 — Key Assumptions (8 marks)

**Paper**: *Gaussian Processes for Time-Marked Time-Series Data*  
**Authors**: John P. Cunningham, Zoubin Ghahramani, Carl E. Rasmussen  
**Venue**: AISTATS 2012  
**Roll Number**: 230035 — Karthik Reddy

## Assumption 1: Time Markers Are Known Exactly and Given A Priori

**Assumption**: The time-marked GP model assumes that all event markers $\{m_k^{(n)}\}$ are precise, noise-free quantities that are known before inference begins. The entire input representation — each observation's (K+1)-dimensional vector of relative times — is constructed from these marker values, so any error in the markers directly corrupts the input space.

**Why the method needs it**: The time-marked covariance kernel (Eq. 2) computes distances in the marker-relative input space. If marker times are uncertain or noisy, the computed covariances no longer reflect the true similarity between observations. More critically, the causal GP warping $h(t) = \max(0, t - m_k)$ creates a hard boundary at each marker — even a small error in $m_k$ shifts this boundary, causing the model to either include pre-event noise (if $m_k$ is too late) or cut off the early response (if $m_k$ is too early).

**Violation scenario**: In clinical EEG data, the "event" might be the onset of a seizure, which is identified retrospectively by a neurologist and can vary by seconds depending on who annotates it. In such settings, marker times are inherently uncertain, and the model's hard causal boundary would be misplaced.

**Paper reference**: Section 2, Equation 1 — the full model conditions on exact marker values. The paper acknowledges this implicitly in Section 4 (Discussion), where it mentions that treating markers as latent parameters is "outside the scope of this work."

## Assumption 2: The Signal's Response to Each Marker Is Separable Across Dimensions

**Assumption**: By using a product-form squared exponential kernel over the multi-dimensional input space (Eq. 2), the model assumes that the signal's dependency on each marker can be decomposed into independent, multiplicative contributions. In other words, how the signal responds to marker 1 does not change depending on marker 2 — the interactions between markers are captured only through their geometric distance in the input space, not through explicit interaction terms.

**Why the method needs it**: The separable kernel structure is what makes the model tractable and allows standard GP inference. Each lengthscale $l_k$ independently controls the influence of one marker. If marker interactions were truly non-separable (e.g., the response to marker 1 depends on how close marker 2 is), the SE kernel would need to be replaced with a more complex, interaction-aware kernel, potentially losing analytical tractability.

**Violation scenario**: In a drug interaction study, the response to a second drug dose might depend critically on how long ago the first dose was administered (pharmacokinetic interactions). The multiplicative SE kernel cannot model such conditional dependencies between markers — it would mis-estimate the joint effect.

**Paper reference**: Equation 2 (Section 2) — the kernel is a product of per-dimension exponential terms with separate lengthscales. The paper does not discuss non-separable kernel alternatives.

## Assumption 3: The Signal Is Constant (Flat) Before Each Event in the Causal Model

**Assumption**: The causal GP enforces that, for a given sample path, $y(t_1) = y(t_2)$ whenever the positive part $(t_1)_+ = (t_2)_+$. In the time-marked context, this means the signal is literally flat (constant) in every direction where the relative time is negative — the signal cannot change before a marker fires. This is a strong structural constraint: there is no "ramp-up" or "anticipation" allowed.

**Why the method needs it**: The causal warping $h(t) = t \cdot \mathbb{I}(t > 0)$ is what creates this flatness. Without it, the GP would see pre-event and post-event observations as having different relative times and would try to fit the pre-event region, potentially overfitting noise. The flatness constraint acts as a structural prior that reduces model complexity in the pre-event region and forces the model to assign pre-event variability to noise.

**Violation scenario**: In financial markets, asset prices often move *before* a scheduled event (e.g., earnings announcement) due to information leakage, insider trading, or anticipatory speculation. In this scenario, the pre-event signal is genuinely non-flat, and forcing it to be constant would prevent the model from capturing meaningful pre-event trends.

**Paper reference**: Section 2.1 — the formal definition of the causal GP states $\frac{\partial y(t)}{\partial t_k} = 0 \ \forall\ t_k < 0$. The paper acknowledges this is appropriate "in many settings" but not universally (opening paragraph of Section 2.1).

## Assumption 4: Stationarity of the Underlying Response Process

**Assumption**: The base kernel used in the time-marked GP is stationary (specifically, a squared exponential). This means the method assumes that the *shape* and *scale* of the signal's response to each marker is the same regardless of when in absolute time the event occurs. The response profile to marker $k$ is always governed by the same lengthscale $l_k$, whether it happens at the start or end of the observation period.

**Why the method needs it**: Stationarity in the base kernel is what allows the model to pool data across all N time series for learning. If the response to marker $k$ in trial $n=1$ had a fundamentally different shape than in trial $n=15$, the model could not use shared lengthscales to generalize across trials. The stationarity assumption is essential for the cross-trial learning that makes the time-marked GP powerful.

**Violation scenario**: In a longitudinal rehabilitation study, a patient's motor response to a stimulus may change significantly over weeks as they recover — early sessions might produce slow, weak responses while later sessions produce fast, strong ones. Using a single set of lengthscales across all sessions would blur these differences.

**Paper reference**: Section 2, Equation 2 — the SE kernel $k_{TM}$ is stationary by construction (depends only on differences). The paper notes in Section 4 (Discussion) that the model can be composed with existing GP technologies, but does not address non-stationarity explicitly.